In [68]:
import folium
import numpy as np
import pandas as pd
import geopandas as gpd

geo_data = gpd.read_file("geo.zip")

geo_data = geo_data.drop(
    ["id", "timestamp", "SRID", "admin_level", "rpath", "tags"],
    axis=1,
    errors="ignore"
)

if geo_data.crs != "EPSG:4326":
    geo_data = geo_data.to_crs("EPSG:4326")

if "localname" in geo_data.columns:
    district_col = "localname"
else:
    district_col = "name"

geo_data = geo_data[[district_col, "geometry"]]

np.random.seed(42)

n = 100
center = [10.7769, 106.7009]

latitude = center[0] + np.random.normal(0, 0.03, n)
longitude = center[1] + np.random.normal(0, 0.03, n)

don_hang = np.random.randint(20, 150, n)
doanh_thu = don_hang * np.random.randint(80000, 250000, n)

df = pd.DataFrame({
    "latitude": latitude,
    "longitude": longitude,
    "don_hang": don_hang,
    "doanh_thu": doanh_thu
})

gdf_points = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326"
)

joined = gpd.sjoin(
    gdf_points,
    geo_data,
    how="left",
    predicate="within"
)

joined.rename(columns={district_col: "District"}, inplace=True)
joined["District"] = joined["District"].fillna("Không xác định")

summary = joined.groupby("District").agg(
    Tong_don_hang=("don_hang", "sum"),
    Tong_doanh_thu=("doanh_thu", "sum"),
    Don_hang_TB=("don_hang", "mean"),
    Doanh_thu_TB=("doanh_thu", "mean")
).reset_index()

summary["Rank_Revenue"] = summary["Tong_doanh_thu"].rank(
    ascending=False,
    method="dense"
).astype(int)

summary["Rank_Orders"] = summary["Tong_don_hang"].rank(
    ascending=False,
    method="dense"
).astype(int)

avg_rev = summary["Tong_doanh_thu"].mean()
avg_order = summary["Tong_don_hang"].mean()

def nhan_xet(row):
    if row["Tong_doanh_thu"] >= avg_rev and row["Tong_don_hang"] >= avg_order:
        return "Hiệu quả cao"
    elif row["Tong_doanh_thu"] >= avg_rev and row["Tong_don_hang"] < avg_order:
        return "Giá trị đơn hàng cao"
    elif row["Tong_doanh_thu"] < avg_rev and row["Tong_don_hang"] >= avg_order:
        return "Nhiều đơn nhưng doanh thu thấp"
    else:
        return "Cần cải thiện"

summary["Nhan_xet"] = summary.apply(nhan_xet, axis=1)
summary = summary.sort_values(by="Tong_doanh_thu", ascending=False)

m = folium.Map(location=center, zoom_start=11)

colors = [
    "#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4",
    "#F7DC6F", "#BB8FCE", "#F1948A", "#5DADE2",
    "#58D68D", "#F5B041", "#EC7063", "#AF7AC5",
    "#48C9B0", "#5499C7", "#F4D03F", "#85C1E9",
    "#F8C471", "#73C6B6", "#E59866", "#D7BDE2"
]

district_names = geo_data[district_col].dropna().unique()

color_dict = {}

for i in range(len(district_names)):
    district = district_names[i]
    color_dict[district] = colors[i % len(colors)]

folium.GeoJson(
    geo_data,
    name="Các quận TP.HCM",
    style_function=lambda feature: {
        "fillColor": color_dict.get(feature["properties"][district_col], "gray"),
        "color": "black",
        "weight": 1,
        "fillOpacity": 0.45
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[district_col],
        aliases=["Quận:"]
    )
).add_to(m)

fg_points = folium.FeatureGroup(name="Điểm dữ liệu đơn hàng")

for i in range(len(joined)):
    row = joined.iloc[i]

    popup_text = ""
    popup_text += "<b>Quận</b>: " + str(row["District"]) + "<br>"
    popup_text += "<b>Đơn hàng</b>: " + str(row["don_hang"]) + "<br>"
    popup_text += "<b>Doanh thu</b>: " + str(row["doanh_thu"]) + "<br>"

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=5,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.8,
        popup=folium.Popup(popup_text, max_width=300),
        tooltip="Click vào"
    ).add_to(fg_points)

fg_points.add_to(m)

fg_summary = folium.FeatureGroup(name="Nhận xét theo quận")

for i in range(len(summary)):
    district = summary.loc[i, "District"]

    if district != "Không xác định":
        khu = geo_data[geo_data[district_col] == district]

        if len(khu) > 0:
            tam = khu.geometry.centroid.iloc[0]

            popup_text = ""
            popup_text += "<b>Quận</b>: " + str(summary.loc[i, "District"]) + "<br>"
            popup_text += "<b>Tổng đơn hàng</b>: " + str(int(summary.loc[i, "Tong_don_hang"])) + "<br>"
            popup_text += "<b>Tổng doanh thu</b>: " + str(int(summary.loc[i, "Tong_doanh_thu"])) + "<br>"
            popup_text += "<b>Đơn hàng trung bình</b>: " + str(round(summary.loc[i, "Don_hang_TB"], 2)) + "<br>"
            popup_text += "<b>Doanh thu trung bình</b>: " + str(round(summary.loc[i, "Doanh_thu_TB"], 2)) + "<br>"
            popup_text += "<b>Hạng doanh thu</b>: " + str(int(summary.loc[i, "Rank_Revenue"])) + "<br>"
            popup_text += "<b>Hạng đơn hàng</b>: " + str(int(summary.loc[i, "Rank_Orders"])) + "<br>"
            popup_text += "<b>Nhận xét</b>: " + str(summary.loc[i, "Nhan_xet"])

            folium.Marker(
                location=[tam.y, tam.x],
                popup=folium.Popup(popup_text, max_width=350),
                tooltip=str(district)).add_to(fg_summary)

fg_summary.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m

/tmp/ipykernel_45014/2675787267.py:156: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  tam = khu.geometry.centroid.iloc[0]
/tmp/ipykernel_45014/2675787267.py:156: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  tam = khu.geometry.centroid.iloc[0]
/tmp/ipykernel_45014/2675787267.py:156: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  tam = khu.geometry.centroid.iloc[0]
/tmp/ipykernel_45014/2675787267.py:156: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before thi